In [1]:
import sqlite3
import pandas as pd

# connect to the same database we created earlier
conn = sqlite3.connect('../data/processed/superstore.db')

# quick check that we're connected properly
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product Name,Sales,Quantity,Discount,Profit,OrderMonth,Profit Margin,Shipping Duration,Order Size,Is Loss
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016-11,16.00,3,Large,0
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016-11,30.00,3,Large,0
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016-06,47.00,4,Small,0
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015-10,-40.00,7,Large,1
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015-10,11.25,7,Small,0


In [2]:
query1 = """
SELECT Region,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct
FROM orders
GROUP BY Region
ORDER BY Total_Profit DESC;
"""
pd.read_sql(query1, conn)

,Region,Total_Sales,Total_Profit,Profit_Margin_Pct
0,West,725457.82,108418.45,14.94
1,East,678781.24,91522.78,13.48
2,South,391721.91,46749.43,11.93
3,Central,501239.89,39706.36,7.92


In [3]:
# Key Insight / Note on this query
# GROUP BY Region aggregates all orders per region into single summary rows
# ROUND() keeps the output clean and readable, avoiding long decimal numbers
# Profit_Margin_Pct is calculated directly in SQL, showing profit efficiency
# not just raw profit dollars, which lets us compare regions fairly

In [4]:
query2 = """
SELECT Discount,
       ROUND(AVG(Profit), 2) AS Avg_Profit,
       COUNT(*) AS Num_Orders
FROM orders
GROUP BY Discount
ORDER BY Discount;
"""
pd.read_sql(query2, conn)

,Discount,Avg_Profit,Num_Orders
0,0.00,66.90,4798
1,0.10,96.06,94
2,0.15,27.29,52
3,0.20,24.70,3657
4,0.30,-45.68,227
5,0.32,-88.56,27
6,0.40,-111.93,206
7,0.45,-226.65,11
8,0.50,-310.70,66
9,0.60,-43.08,138


In [5]:
# Key Insight: SQL Results Confirm Python EDA Findings
# Region ranking matches exactly: West strongest (14.94% margin), Central weakest (7.92%)
# Discount tipping point confirmed again: 20% and below stays profitable,
# 30% and above turns consistently negative
# Cross-validating findings across both Python and SQL strengthens confidence
# that these are real patterns in the data, not calculation errors

In [6]:
query3 = """
SELECT "Sub-Category",
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(AVG(Discount), 3) AS Avg_Discount
FROM orders
GROUP BY "Sub-Category"
ORDER BY Total_Profit ASC
LIMIT 5;
"""
pd.read_sql(query3, conn)

,Sub-Category,Total_Profit,Avg_Discount
0,Tables,-17725.48,0.261
1,Bookcases,-3472.56,0.211
2,Supplies,-1189.10,0.077
3,Fasteners,949.52,0.082
4,Machines,3384.76,0.306


In [7]:
query4 = """
SELECT OrderMonth,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM orders
GROUP BY OrderMonth
ORDER BY OrderMonth;
"""
pd.read_sql(query4, conn)

,OrderMonth,Total_Sales,Total_Profit
0,2014-01,14236.90,2450.19
1,2014-02,4519.89,862.31
2,2014-03,55691.01,498.73
3,2014-04,28295.35,3488.84
4,2014-05,23648.29,2738.71
5,2014-06,34595.13,4976.52
6,2014-07,33946.39,-841.48
7,2014-08,27909.47,5318.10
8,2014-09,81777.35,8328.10
9,2014-10,31453.39,3448.26
